# Train Models

Train or load the configured base and ensemble models, save artifacts, and show one metrics table.

Run `03_analyze.ipynb` for plots and feature importance.


In [ ]:
from pathlib import Path
import sys
def ensure_project_root() -> Path:
    """Put the repo root on sys.path so `import pricing_lab` works without pip install."""
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        if (root / "pricing_lab" / "__init__.py").is_file():
            s = str(root)
            if s not in sys.path:
                sys.path.insert(0, s)
            return root
    raise RuntimeError(
        "Could not find project root (folder containing pricing_lab). "
        "Open/run this notebook from the BNB_Pricing repo or set PYTHONPATH."
    )


!python --version

PROJECT_ROOT = ensure_project_root()
print("Project root:", PROJECT_ROOT)

In [ ]:
# Configure one training run here.
CSV_PATH = None
MODE = "sample"  # sample | full
RUN_TRAINING = True

# Trial overrides; None uses the selected mode defaults.
N_TRIALS_ELASTIC = None
N_TRIALS_KNN = None
N_TRIALS_XGB = None
N_TRIALS_RANDOM_FOREST = None
N_TRIALS_SVM = None
N_TRIALS_NEURAL_NETWORK = None
N_TRIALS_WEIGHTED_ENSEMBLE = None

# Outputs
OUTPUT_CSV = None
SAVE_MODELS = True
MODEL_OUTPUT_DIR = "artifacts"

# Per-model retraining flags; loaded artifacts are reused when False.
RETRAIN_ELASTICNET = False
RETRAIN_KNN = False
RETRAIN_XGBOOST = False
RETRAIN_RANDOM_FOREST = False
RETRAIN_SVM = False
RETRAIN_NEURAL_NETWORK = True
RETRAIN_VOTING_EQUAL = False
RETRAIN_VOTING_WEIGHTED = False
RETRAIN_STACKING = False

In [ ]:
import json
import warnings
from typing import Any

import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from joblib import dump, load
from sklearn.exceptions import ConvergenceWarning

from pricing_lab import config
from pricing_lab.data import TrainTestData, load_train_test
from pricing_lab.metrics import compute_dollar_metrics
from pricing_lab.models.elastic_net import tune_elastic_net
from pricing_lab.models.ensemble import (
    fit_equal_voting_ensemble,
    fit_stacking_ensemble,
    fit_weighted_voting_ensemble,
    select_ensemble_candidates,
)
from pricing_lab.models.knn import tune_knn
from pricing_lab.models.neural_network import tune_neural_network
from pricing_lab.models.random_forest import tune_random_forest
from pricing_lab.models.svm import tune_svm
from pricing_lab.models.xgboost_model import tune_xgboost

SAMPLE_TRAIN_FRACTION = 0.35
SAMPLE_TRIALS = {
    "elastic_net": 10,
    "knn": 6,
    "xgboost": 14,
    "random_forest": 10,
    "svm": 8,
    "neural_network": 12,
    "weighted_ensemble": 8,
}
FULL_TRIALS = {
    "elastic_net": config.N_TRIALS_ELASTICNET,
    "knn": config.N_TRIALS_KNN,
    "xgboost": config.N_TRIALS_XGBOOST,
    "random_forest": config.N_TRIALS_RANDOM_FOREST,
    "svm": config.N_TRIALS_SVM,
    "neural_network": config.N_TRIALS_NEURAL_NETWORK,
    "weighted_ensemble": 20,
}

trial_overrides = {
    "elastic_net": N_TRIALS_ELASTIC,
    "knn": N_TRIALS_KNN,
    "xgboost": N_TRIALS_XGB,
    "random_forest": N_TRIALS_RANDOM_FOREST,
    "svm": N_TRIALS_SVM,
    "neural_network": N_TRIALS_NEURAL_NETWORK,
    "weighted_ensemble": N_TRIALS_WEIGHTED_ENSEMBLE,
}

resolved = config.DATA_PATH if CSV_PATH is None else Path(CSV_PATH)
print("Using CSV:", resolved)
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

full_data: TrainTestData = load_train_test(csv_path=str(CSV_PATH) if CSV_PATH is not None else None)
print(f"Mode: {MODE}")
print(f"Train rows: {len(full_data.X_train)}, test rows: {len(full_data.X_test)}")

if MODE not in {"sample", "full"}:
    raise ValueError("MODE must be 'sample' or 'full'.")

if MODE == "sample":
    sampled_features = full_data.X_train.sample(frac=SAMPLE_TRAIN_FRACTION, random_state=config.RANDOM_STATE)
    sampled_target = full_data.y_train.loc[sampled_features.index]
    data = TrainTestData(
        X_train=sampled_features.reset_index(drop=True),
        X_test=full_data.X_test,
        y_train=sampled_target.reset_index(drop=True),
        y_test=full_data.y_test,
    )
    print(f"Sample mode uses {len(data.X_train)} / {len(full_data.X_train)} train rows ({SAMPLE_TRAIN_FRACTION:.0%}).")
else:
    data = full_data


def resolve_trials(key: str) -> int:
    override = trial_overrides[key]
    if override is not None:
        return int(override)
    defaults = SAMPLE_TRIALS if MODE == "sample" else FULL_TRIALS
    return int(defaults[key])


model_output_path = PROJECT_ROOT / MODEL_OUTPUT_DIR
model_output_path.mkdir(parents=True, exist_ok=True)

base_specs = [
    ("ElasticNet", "elastic_net.joblib", RETRAIN_ELASTICNET, lambda: tune_elastic_net(data, n_trials=resolve_trials("elastic_net"))),
    ("KNN", "knn.joblib", RETRAIN_KNN, lambda: tune_knn(data, n_trials=resolve_trials("knn"))),
    ("XGBoost", "xgboost.joblib", RETRAIN_XGBOOST, lambda: tune_xgboost(data, n_trials=resolve_trials("xgboost"))),
    ("RandomForest", "random_forest.joblib", RETRAIN_RANDOM_FOREST, lambda: tune_random_forest(data, n_trials=resolve_trials("random_forest"))),
    ("SVM", "svm.joblib", RETRAIN_SVM, lambda: tune_svm(data, n_trials=resolve_trials("svm"))),
    ("NeuralNetwork", "neural_network.joblib", RETRAIN_NEURAL_NETWORK, lambda: tune_neural_network(data, n_trials=resolve_trials("neural_network"))),
]

model_pipelines: dict[str, Any] = {}
metrics_map: dict[str, dict[str, Any]] = {}

for model_name, artifact_name, retrain_flag, train_fn in base_specs:
    artifact_path = model_output_path / artifact_name
    should_retrain = RUN_TRAINING and retrain_flag
    if should_retrain:
        print(f"Training {model_name}...")
        result = train_fn()
        model_pipelines[model_name] = result.pipeline
        metrics_map[model_name] = {
            "name": result.name,
            "cv_rmse_log": float(result.best_cv_rmse_log),
            "test_metrics": result.test_metrics,
            "best_params": result.best_params,
        }
        if SAVE_MODELS:
            dump(result.pipeline, artifact_path)
            print(f"Wrote {artifact_path}")
        continue
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing artifact for {model_name}: {artifact_path}. Enable RETRAIN flag or RUN_TRAINING.")
    model_pipelines[model_name] = load(artifact_path)

# Re-evaluate loaded artifacts on the current test split.
y_true_dollars = np.expm1(data.y_test.to_numpy(dtype=np.float64))


def evaluate_pipeline(model_name: str, pipeline: Any) -> dict[str, Any]:
    y_pred_log = np.asarray(pipeline.predict(data.X_test), dtype=np.float64)
    test_metrics = compute_dollar_metrics(data.y_test.to_numpy(dtype=np.float64), y_pred_log)
    return {
        "name": model_name,
        "cv_rmse_log": float("nan"),
        "test_metrics": test_metrics,
        "best_params": {"source": "loaded_artifact"},
    }


for model_name, _, _, _ in base_specs:
    if model_name not in metrics_map:
        metrics_map[model_name] = evaluate_pipeline(model_name, model_pipelines[model_name])

# Base models must exist before ensemble models are fit or loaded.
base_selection_scores: dict[str, float] = {}
for model_name in [spec[0] for spec in base_specs]:
    cv_score = float(metrics_map[model_name]["cv_rmse_log"])
    if np.isfinite(cv_score):
        base_selection_scores[model_name] = cv_score
    else:
        base_selection_scores[model_name] = float(metrics_map[model_name]["test_metrics"]["rmse"])

selected_base = select_ensemble_candidates(
    {name: model_pipelines[name] for name in base_selection_scores},
    base_selection_scores,
)
print(f"Ensemble candidates: {', '.join(selected_base.keys())}")

ensemble_specs = [
    ("VotingEnsembleEqual", "voting_equal.joblib", RETRAIN_VOTING_EQUAL),
    ("VotingEnsembleWeighted", "voting_weighted.joblib", RETRAIN_VOTING_WEIGHTED),
    ("StackingEnsemble", "stacking.joblib", RETRAIN_STACKING),
]

for ensemble_name, artifact_name, retrain_flag in ensemble_specs:
    artifact_path = model_output_path / artifact_name
    should_retrain = RUN_TRAINING and retrain_flag
    if should_retrain:
        print(f"Training {ensemble_name}...")
        if ensemble_name == "VotingEnsembleEqual":
            result = fit_equal_voting_ensemble(data, selected_base)
        elif ensemble_name == "VotingEnsembleWeighted":
            result = fit_weighted_voting_ensemble(data, selected_base, n_trials=resolve_trials("weighted_ensemble"))
        else:
            result = fit_stacking_ensemble(data, selected_base)
        model_pipelines[ensemble_name] = result.pipeline
        metrics_map[ensemble_name] = {
            "name": result.name,
            "cv_rmse_log": float(result.best_cv_rmse_log),
            "test_metrics": result.test_metrics,
            "best_params": result.best_params,
        }
        if SAVE_MODELS:
            dump(result.pipeline, artifact_path)
            print(f"Wrote {artifact_path}")
        continue
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing artifact for {ensemble_name}: {artifact_path}. Enable RETRAIN flag or RUN_TRAINING.")
    loaded_pipeline = load(artifact_path)
    model_pipelines[ensemble_name] = loaded_pipeline
    metrics_map[ensemble_name] = evaluate_pipeline(ensemble_name, loaded_pipeline)


def validate_metric_payload(model_name: str, test_metrics: dict[str, float]) -> None:
    mae_value = float(test_metrics["mae"])
    rmse_value = float(test_metrics["rmse"])
    if rmse_value + 1e-12 < mae_value:
        raise ValueError(f"Invalid metrics for {model_name}: RMSE must be greater than or equal to MAE.")


def result_row(payload: dict[str, Any]) -> dict[str, Any]:
    cv_value = float(payload["cv_rmse_log"])
    cv_output = round(cv_value, 6) if np.isfinite(cv_value) else float("nan")
    model_name = str(payload["name"])
    test_metrics = payload["test_metrics"]
    validate_metric_payload(model_name, test_metrics)
    return {
        "model": model_name,
        "cv_rmse_log": cv_output,
        "test_mae_dollars": round(float(test_metrics["mae"]), 4),
        "test_rmse_dollars": round(float(test_metrics["rmse"]), 4),
        "test_r2": round(float(test_metrics["r2"]), 6),
        "best_params_json": json.dumps(payload["best_params"], sort_keys=True),
    }


rows = [result_row(metrics_map[name]) for name in model_pipelines.keys()]
table = pd.DataFrame(rows)
display(table)
print(table.to_string(index=False))

if OUTPUT_CSV is not None:
    output_path = PROJECT_ROOT / OUTPUT_CSV
    output_path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_path, index=False)
    print(f"Wrote {output_path}")